# CASMI 2026: Data First — Learn to Read Spectra, Then Explore

This notebook is for readers new to mass spectrometry and competition data. Follow one chain of questions:

**Read one spectrum → distinguish a row from a molecule → identify usable fields → inspect repeated structures and sources → audit quality and filtering → choose the next check.**

Each section asks a question, shows data, explains how to read the figure, and separates the observation from its implication. Detailed column definitions and audit CSVs come last.

**Task:** combine all spectra for a `molecule_id` into one ranked structure list. Training SMILES are known answers. Visible test is a placeholder drawn from training examples, useful for interface checks.
[Official data description](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/data)

In [ ]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

SEED = 2026
SPECTRA_SAMPLE_N = 4096
USE_SAMPLE_CACHE = True

DATA_DIR_OVERRIDE = None
required = ('train.parquet', 'test.parquet', 'sample_submission.csv')
if DATA_DIR_OVERRIDE is not None:
    candidates = [Path(DATA_DIR_OVERRIDE).expanduser()]
elif Path('/kaggle/input').exists():
    candidates = [p.parent for p in Path('/kaggle/input').rglob('train.parquet')]
else:
    candidates = [Path.cwd() / 'data', Path.cwd().parent / 'data', Path.cwd()]
matches = sorted({p.resolve() for p in candidates if all((p / n).is_file() for n in required)})
if len(matches) != 1:
    raise FileNotFoundError(f'Expected one data directory with all three files; found {matches}')
DATA_DIR = matches[0]
OUTPUT_DIR = (Path('/kaggle/working/eda_evidence') if Path('/kaggle/input').exists()
              else DATA_DIR.parent / 'output' / 'eda-v3')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_file = pq.ParquetFile(DATA_DIR / 'train.parquet')
test_file = pq.ParquetFile(DATA_DIR / 'test.parquet')
train_head = next(train_file.iter_batches(batch_size=5)).to_pandas()
test = test_file.read().to_pandas()  # The visible test file is small.
test_head = test.head(5)
submission = pd.read_csv(DATA_DIR / 'sample_submission.csv', dtype=str)


import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import FancyBboxPatch
from textwrap import fill

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
BLUE, ORANGE, GREEN, GRAY = '#2166ac', '#d6604d', '#238b45', '#6b7280'
figure_records = []

def show_figure(name, fig):
    fig.tight_layout()
    path = OUTPUT_DIR / f'{name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor='white')
    figure_records.append({'name': name, 'file': path.name,
                           'panels': len(fig.axes)})
    plt.show(); plt.close(fig)

# Kaggle's base image may omit RDKit. Install the pinned depiction package only
# when absent; this is for pictures, not official scoring normalization.
if Path('/kaggle/input').exists():
    import importlib.util
    if importlib.util.find_spec('rdkit') is None:
        import subprocess
        import sys
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                        '--disable-pip-version-check', '--no-deps', 'rdkit==2026.9.1'],
                       check=True, timeout=180)

try:
    from rdkit import Chem
    from rdkit.Chem import Draw
    HAS_RDKIT = True
except ImportError:
    HAS_RDKIT = False

def molecule_picture(ax, smiles):
    if HAS_RDKIT:
        molecule = Chem.MolFromSmiles(str(smiles))
        if molecule is not None:
            ax.imshow(Draw.MolToImage(molecule, size=(500, 300)))
        else:
            ax.text(.5, .5, 'Structure parsing unavailable', ha='center', transform=ax.transAxes)
    else:
        ax.text(.5, .5, fill(str(smiles), 45), ha='center', va='center', transform=ax.transAxes,
                fontsize=9)
        ax.text(.5, .05, 'SMILES shown; optional RDKit is not installed', ha='center', transform=ax.transAxes, fontsize=8)
    ax.set_axis_off()

display(pd.DataFrame([
    {'file': 'train.parquet', 'rows': train_file.metadata.num_rows, 'unit': 'spectrum + known structure'},
    {'file': 'test.parquet', 'rows': len(test), 'unit': 'spectrum; visible placeholder'},
    {'file': 'sample_submission.csv', 'rows': len(submission), 'unit': 'one molecule prediction'},
]))

## 1. What are we predicting?

Start with the information flow. **A spectrum is a measurement; a structure is the answer to infer.**
The spectrum's horizontal axis is fragment mass-to-charge ratio `m/z`, and its vertical axis is relative intensity. It is not a time series.
Different adducts, collision energies or repeated measurements can produce several spectra for one molecule.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.1))
ax.set(xlim=(0, 13), ylim=(0, 3)); ax.axis('off')
steps = [('Unknown molecule', 'One target molecule', '#e8f0fa'),
         ('Measure ions', 'Adduct + collision energy', '#e8f5ec'),
         ('Read spectra', 'Paired m/z and intensity arrays', '#e8f0fa'),
         ('Rank structures', 'Combine spectra; one list per molecule', '#fff0e8')]
for i, (title, subtitle, color) in enumerate(steps):
    x = .15+i*3.25
    ax.add_patch(FancyBboxPatch((x, .9), 2.8, 1.3, boxstyle='round,pad=.08', facecolor=color, edgecolor='#b6c2cf'))
    ax.text(x+1.4, 1.8, title, ha='center', fontsize=12, weight='bold')
    ax.text(x+1.4, 1.25, fill(subtitle, 27), ha='center', va='center', fontsize=10)
    if i < 3:
        ax.annotate('', xy=(x+3.13, 1.55), xytext=(x+2.89, 1.55), arrowprops={'arrowstyle':'->', 'color':GRAY, 'lw':1.5})
ax.text(6.5, .35, 'Concept diagram: inputs are spectra; the prediction and scoring unit is a molecule.', ha='center', color=GRAY)
show_figure('01_task_flow', fig)

### Read one real training spectrum: one array position pairs two values

The left panel depicts a known training SMILES, the middle panel shows its spectrum, and the right panel exposes three strong peaks and their array positions.
**Read it:** each vertical line is a fragment peak; height gives relative intensity. Keep m/z and intensity paired by index—never sort them separately.
The molecule picture displays a training answer; it is not a test input or an implementation of scoring normalization.

In [ ]:
illustration = train_head.iloc[train_head['ms2_mzs'].map(len).argmax()]
mz = np.asarray(illustration['ms2_mzs'], dtype=float)
intensity = np.asarray(illustration['ms2_normalized_intensities'], dtype=float)
assert len(mz) == len(intensity)
strong_positions = np.argsort(intensity)[-min(3, len(mz)):][::-1]
fig, axes = plt.subplots(1, 3, figsize=(14, 4), gridspec_kw={'width_ratios':[1.1, 1.5, 1]})
molecule_picture(axes[0], illustration['normalized_smiles'])
axes[0].set_title('Known training structure')
axes[1].vlines(mz, 0, intensity, color=BLUE, linewidth=1)
axes[1].set(xlabel='Fragment m/z', ylabel='Relative intensity', ylim=(0, 1.15),
            title=f"Real spectrum | {illustration['adduct']} | {len(mz)} peaks")
for position, text_x in zip(strong_positions, [.42, .73, .1]):
    axes[1].annotate(f'array[{position}]', (mz[position], intensity[position]), xytext=(text_x, .96),
                     textcoords='axes fraction', ha='center', fontsize=8,
                     arrowprops={'arrowstyle':'-', 'color':GRAY, 'lw':.7})
axes[2].axis('off'); axes[2].set_title('The same three paired array positions')
for j, position in enumerate(strong_positions):
    axes[2].text(.02, .8-j*.22, f'index {position}\nm/z = {mz[position]:.4f}\nintensity = {intensity[position]:.3f}',
                 transform=axes[2].transAxes, fontsize=10, va='center')
show_figure('02_molecule_and_paired_peaks', fig)

## 2. Is a row a spectrum or an independent molecule?

Read scalar metadata for all training rows, then choose a real structure repeated in the first 10,000 rows.
Plot several of its spectra side by side. This is an illustrative group, not a representative sample of the training population.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

meta_columns = ['normalized_smiles', 'inchikey14', 'instrument_type', 'adduct',
                'num_peaks', 'ingest_lib', 'molecular_formula', 'precursor_mz',
                'precursor_error_ppm', 'base_peak_intensity', 'ionization_mode',
                'collision_energy_orig_units']
meta = train_file.read(columns=meta_columns).to_pandas(types_mapper=pd.ArrowDtype)
assert len(meta) == train_file.metadata.num_rows
train_groups = meta['normalized_smiles'].value_counts()
test_groups = test['molecule_id'].value_counts()
train_connectivity_groups = meta['inchikey14'].nunique()
train_peaks = pd.to_numeric(meta['num_peaks'], errors='coerce').dropna().to_numpy(dtype=float)
test_peaks = test['ms2_mzs'].map(len).to_numpy(dtype=float)

In [ ]:
prefix = meta.iloc[:10000]
repeated_smiles = prefix['normalized_smiles'].value_counts().index[0]
demo_indices = prefix.index[prefix['normalized_smiles'].eq(repeated_smiles).fillna(False)][:4].to_numpy()
demo_parts, offset = [], 0
demo_cols = ['ms2_mzs', 'ms2_normalized_intensities', 'collision_energy_ev']
for batch in train_file.iter_batches(batch_size=512, columns=demo_cols):
    lo, hi = np.searchsorted(demo_indices, [offset, offset+len(batch)])
    if hi > lo:
        chosen = batch.take(__import__('pyarrow').array(demo_indices[lo:hi]-offset))
        demo_parts.append(chosen.to_pandas())
    offset += len(batch)
    if offset > demo_indices[-1]:
        break
demo_spectra = pd.concat(demo_parts, ignore_index=True)
demo_metadata = meta.iloc[demo_indices].reset_index(drop=True)
fig, axes = plt.subplots(2, 2, figsize=(12, 6.4))
for ax, i in zip(axes.flat, range(4)):
    if i >= len(demo_spectra):
        ax.axis('off'); continue
    row, info = demo_spectra.iloc[i], demo_metadata.iloc[i]
    ax.vlines(row['ms2_mzs'], 0, row['ms2_normalized_intensities'], color=BLUE, linewidth=.8)
    ax.set(title=f"row {demo_indices[i]} | {info['adduct']} | CE {row['collision_energy_ev']}",
           xlabel='Fragment m/z', ylabel='Relative intensity', ylim=(0, 1.08))
fig.suptitle(f"Same known structure, several measurements | {demo_metadata.iloc[0]['inchikey14']}", fontsize=13)
show_figure('03_same_structure_multiple_spectra', fig)
display(pd.DataFrame([
    {'set': 'train', 'spectrum rows': len(meta), 'SMILES labels': len(train_groups), 'connectivity keys': train_connectivity_groups},
    {'set': 'visible placeholder test', 'spectrum rows': len(test), 'SMILES labels': np.nan, 'connectivity keys': np.nan},
]))
print(f"Visible test groups: {len(test_groups)} molecule_id values; {len(test)/len(test_groups):.2f} spectra per molecule.")

**Observation:** one known structure has several measurement rows. Similar-looking spectra do not make those rows independent molecules.
Train connectivity keys organize known structures; test `molecule_id` groups unknown targets. These IDs are not matching join keys.
**Implication:** a model may process individual spectra, but prediction and evaluation must respect the molecule unit.

## 3. Which columns are measurements, and which reveal the answer?

Each row below is a column; each matrix column is a file. A filled tile means the field actually exists.
Group colors distinguish shared measurements, train-only fields and test IDs. **Train-only is not synonymous with target:**
`num_peaks` can be derived from arrays, while `ingest_lib` supports source audits.

In [ ]:
train_columns, test_columns = set(train_file.schema_arrow.names), set(test_file.schema_arrow.names)
shared, train_only, test_only = sorted(train_columns & test_columns), sorted(train_columns-test_columns), sorted(test_columns-train_columns)
fields = shared+train_only+test_only
availability = np.array([[1 if c in train_columns else 0, 1 if c in test_columns else 0] for c in fields])
fig, ax = plt.subplots(figsize=(8.5, 8.2))
ax.imshow(availability, cmap=ListedColormap(['#f1f5f9', BLUE]), vmin=0, vmax=1, aspect='auto')
ax.set(xticks=[0,1], xticklabels=['train.parquet', 'test.parquet'], yticks=np.arange(len(fields)), yticklabels=fields,
       title='Actual field availability: filled = present')
for i, c in enumerate(fields):
    for j in range(2):
        ax.text(j, i, 'present' if availability[i,j] else 'absent', ha='center', va='center', color='white' if availability[i,j] else GRAY, fontsize=8)
    ax.get_yticklabels()[i].set_color(BLUE if c in shared else ORANGE if c in train_only else GREEN)
for boundary in [len(shared)-.5, len(shared)+len(train_only)-.5]:
    ax.axhline(boundary, color='white', linewidth=3)
ax.text(1.02, .84, 'Shared measurements', transform=ax.transAxes, color=BLUE, fontsize=10)
ax.text(1.02, .32, 'Train labels / audit metadata', transform=ax.transAxes, color=ORANGE, fontsize=10)
ax.text(1.02, .03, 'Test grouping IDs', transform=ax.transAxes, color=GREEN, fontsize=10)
show_figure('04_field_availability', fig)

**Rule after reading the figure:** `normalized_smiles` is the answer. Structure keys, formulas and label-dependent `precursor_error_ppm` are not ready-made test inputs to a spectrum encoder.
Compute peak counts consistently as `len(ms2_mzs)` on both sides. IDs track or aggregate records, rather than predict chemistry.
An expandable field reference is at the end.

## 4. How common are repeated structures, and why can row splits mislead?

The left panel counts **connectivity groups**, not spectra. Its final bin includes every group with 10 or more spectra.
The right panel is a labeled thought experiment: assign each row independently to training with probability 80%.
Given one validation spectrum, how likely is a same-structure training partner as group size increases?
This is neither a model score nor an implemented validation split.

In [ ]:
group_audit = meta.groupby('inchikey14', dropna=False).agg(
    spectra=('num_peaks', 'size'), smiles_labels=('normalized_smiles', 'nunique'),
    sources=('ingest_lib', 'nunique'), formulas=('molecular_formula', 'nunique'))
assert int(group_audit['spectra'].sum()) == len(meta)
k = group_audit['spectra'].to_numpy(dtype=float)
row_split_partner_pct = 100 * np.average(1 - np.power(0.2, k-1), weights=k)
multi_source_pct = 100 * group_audit['sources'].gt(1).mean()
group_summary = pd.DataFrame([
    {'measure': 'Connectivity groups (including missing-key group)', 'value': len(group_audit)},
    {'measure': 'Rows missing connectivity key', 'value': int(meta['inchikey14'].isna().sum())},
    {'measure': 'Groups with multiple spectra (%)', 'value': 100*group_audit['spectra'].gt(1).mean()},
    {'measure': 'Groups spanning multiple sources (%)', 'value': multi_source_pct},
    {'measure': 'Groups with multiple SMILES labels', 'value': int(group_audit['smiles_labels'].gt(1).sum())},
    {'measure': 'Groups with multiple formulas', 'value': int(group_audit['formulas'].gt(1).sum())},
    {'measure': 'Expected same-structure partner under IID 80/20 rows (%)', 'value': row_split_partner_pct},
])
ambiguous_keys = group_audit.index[group_audit['smiles_labels'].gt(1)][:2]
label_examples = (meta.loc[meta['inchikey14'].isin(ambiguous_keys),
                  ['inchikey14', 'normalized_smiles', 'molecular_formula', 'ingest_lib']]
                  .drop_duplicates().groupby('inchikey14', group_keys=False).head(4))
formula_keys = group_audit.index[group_audit['formulas'].gt(1)][:2]
formula_examples = (meta.loc[meta['inchikey14'].isin(formula_keys),
                    ['inchikey14', 'normalized_smiles', 'molecular_formula', 'ingest_lib']]
                    .drop_duplicates(['inchikey14', 'molecular_formula']))
print('Examples are label variants to inspect, not automatic corrections.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
group_hist = group_audit['spectra'].clip(upper=10).value_counts().reindex(range(1,11), fill_value=0).div(len(group_audit)).mul(100)
axes[0].bar(range(1,11), group_hist, color=[ORANGE]+[BLUE]*9)
axes[0].set(xticks=range(1,11), xticklabels=[str(n) for n in range(1,10)]+['10+'], xlabel='Spectra per connectivity group',
            ylabel='Connectivity groups (%)', title='Most structures have several spectra')
axes[0].annotate(f"One spectrum: {group_hist.iloc[0]:.1f}%", xy=(1, group_hist.iloc[0]), xytext=(2, group_hist.max()*.9),
                 arrowprops={'arrowstyle':'->', 'color':ORANGE}, color=ORANGE)
sizes = np.arange(1, 11)
probability = (1-.2**(sizes-1))*100
axes[1].plot(sizes, probability, 'o-', color=ORANGE)
axes[1].set(xticks=[1,2,3,5,10], xlabel='Spectra for the same structure (k)', ylabel='Chance of a training partner (%)',
            title='IID 80/20 rows: illustrative overlap risk', ylim=(-5, 105))
axes[1].text(.28, .12, f'Weighted across actual group sizes:\n{row_split_partner_pct:.2f}% expected same-structure partners',
             transform=axes[1].transAxes, fontsize=10, bbox={'facecolor':'#fff0e8','edgecolor':'none','pad':7})
show_figure('05_repeated_groups_and_row_split', fig)

**Interpretation:** repeated measurements differ from exact duplicate arrays. Unseen-structure generalization needs structure groups;
known-structure retrieval needs an explicit reference-spectrum protocol.
The file's `inchikey14` defines this audit; the official pinned scoring normalization has not been rerun.
Next examine label variants, then source overlaps.

### Do multiple SMILES establish mislabeling? Inspect structures first

The left panel counts connectivity groups with multiple SMILES labels or formulas.
The other panels depict two real SMILES sharing a supplied connectivity key.
Pictures help inspect tautomer or protonation representations; they do not replace official scoring-normalization checks.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), gridspec_kw={'width_ratios':[1, 1.15, 1.15]})
counts = [int(group_audit['smiles_labels'].gt(1).sum()), int(group_audit['formulas'].gt(1).sum())]
axes[0].bar(['Multiple\nSMILES', 'Multiple\nformulas'], counts, color=[BLUE, ORANGE])
axes[0].set(ylabel='Connectivity groups', title='Variants require review')
for j, count in enumerate(counts):
    axes[0].text(j, count+max(counts)*.04, f'{count:,}\n({count/len(group_audit)*100:.2f}%)', ha='center', fontsize=9)
axes[0].set_ylim(0, max(counts)*1.25)
variant_lengths = label_examples.assign(length=label_examples['normalized_smiles'].astype('string').str.len()).groupby('inchikey14')['length'].mean()
variant_key = variant_lengths.idxmin() if len(variant_lengths) else None
variants = label_examples.loc[label_examples['inchikey14'].eq(variant_key)].drop_duplicates('normalized_smiles').head(2)
for ax, (_, row) in zip(axes[1:], variants.iterrows()):
    molecule_picture(ax, row['normalized_smiles'])
    ax.set_title(f"Same supplied key\n{row['inchikey14']}", fontsize=10)
show_figure('06_label_variants_as_structures', fig)
display(formula_examples[['inchikey14','molecular_formula','ingest_lib']])

**Observation and limit:** displayed formula examples include charge-state variants. Different strings establish neither different scoring answers nor mislabeling.
The cheapest next check is official normalization of these examples. Preserve original labels rather than relabeling or deleting them automatically.

## 5. Sources and instruments: does matching hardware imply matching data?

Compare each source's share of training spectra with the instrument composition inside that source.
The left panel uses all training spectra as its denominator. Each right-panel row sums to 100% and describes one source.
Retain small libraries rather than showing only the largest ones.

In [ ]:
def numeric(series):
    return pd.to_numeric(series, errors='coerce').to_numpy(dtype=float, na_value=np.nan)

source_rows = []
for source, frame in meta.groupby('ingest_lib', dropna=False):
    peaks, mass, err = numeric(frame['num_peaks']), numeric(frame['precursor_mz']), numeric(frame['precursor_error_ppm'])
    finite_err = np.isfinite(err)
    source_rows.append({'source': str(source), 'spectra': len(frame),
        'connectivity groups': frame['inchikey14'].nunique(),
        'peak p50': np.nanmedian(peaks), 'peak p95': np.nanquantile(peaks, .95),
        'precursor m/z p50': np.nanmedian(mass),
        'negative ion %': 100*frame['ionization_mode'].eq('negative').fillna(False).mean(),
        'base intensity missing %': 100*frame['base_peak_intensity'].isna().mean(),
        'ppm finite n': int(finite_err.sum()),
        '|ppm| > 20 among finite %': 100*np.mean(np.abs(err[finite_err]) > 20) if finite_err.any() else np.nan})
source_summary = pd.DataFrame(source_rows).sort_values('spectra', ascending=False)
assert int(source_summary['spectra'].sum()) == len(meta)
source_cat = meta['ingest_lib'].astype('string').fillna('<missing>')
instrument_cat = meta['instrument_type'].astype('string').fillna('<missing>')
source_instruments = pd.crosstab(source_cat, instrument_cat)
top_instruments = list(dict.fromkeys(instrument_cat.value_counts().head(5).index.tolist()+['<missing>']))
instrument_mix = source_instruments.reindex(columns=top_instruments, fill_value=0).copy()
instrument_mix['Other'] = source_instruments.sum(axis=1)-instrument_mix.sum(axis=1)
instrument_mix = instrument_mix.div(source_instruments.sum(axis=1), axis=0).mul(100)
energy_units = pd.crosstab(source_cat, meta['collision_energy_orig_units'].astype('string').fillna('<missing>'))

fixed_slice = meta.loc[(meta['adduct'].eq('[M+H]+') & meta['precursor_mz'].between(200, 600)).fillna(False)]
conditional_peaks = fixed_slice.groupby('ingest_lib')['num_peaks'].agg(
    spectra='size', peak_p50='median', peak_p95=lambda x: x.quantile(.95)).sort_values('spectra', ascending=False)
np_keys = set(meta.loc[meta['ingest_lib'].eq('enveda-np-examples').fillna(False), 'inchikey14'].dropna())
other_keys = set(meta.loc[meta['ingest_lib'].ne('enveda-np-examples').fillna(False), 'inchikey14'].dropna())
np_overlap_n = len(np_keys & other_keys)
print(f'enveda-np-examples: {len(np_keys)} connectivity keys; {np_overlap_n} also occur in other sources.')
print('Source structure counts overlap and must not be summed as independent structures.')

In [ ]:
source_order = source_summary['source'].tolist()
mix = instrument_mix.reindex(source_order)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.6), gridspec_kw={'width_ratios':[1,1.4]})
share = source_summary.set_index('source').loc[source_order, 'spectra']/len(meta)*100
axes[0].barh(source_order, share, color=BLUE); axes[0].invert_yaxis()
axes[0].set(xlabel='All training spectra (%)', title='How much does each source contribute?')
for y, value in enumerate(share):
    axes[0].text(value+.3, y, f'{value:.2f}%', va='center', fontsize=8)
axes[0].set_xlim(0, max(share)*1.2)
left = np.zeros(len(mix)); colors = ['#2166ac','#d6604d','#238b45','#756bb1','#e6ab02','#bdbdbd','#6b7280']
for i, column in enumerate(mix):
    values = mix[column].to_numpy(dtype=float)
    axes[1].barh(source_order, values, left=left, color=colors[i%len(colors)], label=column)
    left += values
axes[1].invert_yaxis(); axes[1].set(xlim=(0,100), xlabel='Spectra within each source (%)', title='Instrument mix inside each source')
axes[1].legend(loc='upper center', bbox_to_anchor=(.5,-.12), ncol=3, fontsize=8)
show_figure('07_sources_and_instruments', fig)

### Control two conditions before interpreting differences

In the left cumulative plot, y at a chosen x is the percentage of spectra with precursor m/z at or below x.
The displayed range is 0–1200; the denominator still includes all finite precursor values from that source.
The right plot compares peak counts only for **`[M+H]+`, precursor m/z 200–600** training spectra.
This is a pre-specified illustrative slice, not a range selected from visible test.
Separated curves within this slice mean overall mass and adduct composition do not explain everything; they do not identify a causal instrument effect.

In [ ]:
def ecdf(ax, values, label, color):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    x, n = np.unique(values, return_counts=True)
    if len(x):
        ax.step(x, np.cumsum(n)/n.sum()*100, where='post', label=f'{label} (n={len(values):,})', color=color)

selected_sources = ['enveda-180', 'pluskal_ms2', 'riken', 'enveda-np-examples']
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for source, color in zip(selected_sources, [BLUE, ORANGE, '#756bb1', GREEN]):
    rows = meta.loc[meta['ingest_lib'].eq(source).fillna(False)]
    sliced = fixed_slice.loc[fixed_slice['ingest_lib'].eq(source).fillna(False)]
    ecdf(axes[0], numeric(rows['precursor_mz']), source, color)
    ecdf(axes[1], numeric(sliced['num_peaks']), source, color)
axes[0].set(xlabel='Precursor m/z', ylabel='Spectra at or below x (%)', title='Mass profiles differ by source', xlim=(0,1200), ylim=(0,101))
axes[1].set(xlabel='Peaks per spectrum (log scale)', ylabel='Spectra at or below x (%)', title='Peak counts in the fixed adduct + mass slice', xscale='log', ylim=(0,101))
for ax in axes:
    ax.legend(fontsize=8, loc='lower right'); ax.grid(alpha=.15)
show_figure('08_source_profiles_and_conditional_peaks', fig)

**Observation and interpretation:** source differences remain in the conditional slice. Hardware, energy, preprocessing and chemistry can all contribute; these plots cannot separate their effects.
The official description says `enveda-180` matches the test hardware but emphasizes synthetic screening chemistry; `enveda-np-examples` is closer to the target setting but small.
All 250 of its supplied connectivity keys also occur elsewhere, so a source holdout does not ensure structure isolation.
[Source descriptions](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/data)

### Are missing values random? Put sources back into the picture

Each tile shows the percentage of spectra missing a field within that source. Darker means more missing.
`Energy unit unresolved` counts missing values and the literal `unknown`, without treating NCE as eV.
Missing raw intensity can mean a source shipped pre-normalized spectra; it does not establish that those spectra are noise.

In [ ]:
missing_rows = []
for source, rows in meta.groupby('ingest_lib', dropna=False):
    units = rows['collision_energy_orig_units'].astype('string').fillna('<missing>')
    missing_rows.append({'source':str(source),
        'Raw base intensity':100*rows['base_peak_intensity'].isna().mean(),
        'Instrument label':100*rows['instrument_type'].isna().mean(),
        'Adduct':100*rows['adduct'].isna().mean(),
        'Precursor m/z':100*rows['precursor_mz'].isna().mean(),
        'Energy unit unresolved':100*units.isin(['unknown','<missing>']).mean()})
missing_matrix = pd.DataFrame(missing_rows).set_index('source').reindex(source_order)
fig, ax = plt.subplots(figsize=(10.5, 5.8))
im = ax.imshow(missing_matrix.to_numpy(dtype=float), vmin=0, vmax=100, cmap='Blues', aspect='auto')
ax.set(xticks=np.arange(len(missing_matrix.columns)), xticklabels=missing_matrix.columns,
       yticks=np.arange(len(missing_matrix)), yticklabels=missing_matrix.index, title='Missingness / unresolved units depend on source')
plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
for i in range(len(missing_matrix)):
    for j in range(len(missing_matrix.columns)):
        value = float(missing_matrix.iloc[i,j]); ax.text(j,i,f'{value:.1f}%',ha='center',va='center',color='white' if value>55 else '#233242',fontsize=8)
fig.colorbar(im, ax=ax, label='Spectra within source (%)', fraction=.04, pad=.02)
show_figure('09_source_missingness', fig)

## 6. Before auditing arrays, establish what the sample covers

Choose 4,096 rows uniformly without replacement over the complete training file with seed 2026.
Stream peak columns on the first run, and reuse a cache only when input identity and sampling settings match.
This sample has a different purpose from the earlier repeated-structure illustration.
Blue points show population source shares; orange points show sample shares. Sources absent or represented by one row do not support source-specific conclusions.

In [ ]:
import hashlib
import json
import platform
import time
import pyarrow as pa

def file_identity(path):
    stat = path.stat()
    footer_digest = None
    if path.suffix == '.parquet':
        with path.open('rb') as handle:
            handle.seek(-8, 2)
            trailer = handle.read(8)
            if trailer[4:] != b'PAR1':
                raise ValueError(f'Invalid Parquet trailer: {path.name}')
            footer_size = int.from_bytes(trailer[:4], 'little')
            handle.seek(-8-footer_size, 2)
            footer_digest = hashlib.sha256(handle.read(footer_size)+trailer).hexdigest()
    return {'name': path.name, 'bytes': stat.st_size, 'mtime_ns': stat.st_mtime_ns,
            'parquet_footer_sha256': footer_digest}

input_identity = [file_identity(DATA_DIR / name) for name in required]
sample_size = min(SPECTRA_SAMPLE_N, train_file.metadata.num_rows)
sample_ids = np.sort(np.random.default_rng(SEED).choice(len(meta), sample_size, replace=False))
sample_key = {'train_identity': input_identity[0], 'seed': SEED, 'n': sample_size, 'method': 'uniform-row-v1'}
cache_path = OUTPUT_DIR / 'train_spectra_sample.parquet'
cache_key_path = OUTPUT_DIR / 'train_spectra_sample.identity.json'
cached = False
if USE_SAMPLE_CACHE and cache_path.exists() and cache_key_path.exists():
    cached = json.loads(cache_key_path.read_text()) == sample_key
if cached:
    spectra_sample = pd.read_parquet(cache_path)
else:
    selected, offset = [], 0
    started = time.perf_counter()
    columns = ['ms2_mzs', 'ms2_normalized_intensities', 'collision_energy_ev']
    for batch in train_file.iter_batches(batch_size=4096, columns=columns):
        end = offset+len(batch)
        lo, hi = np.searchsorted(sample_ids, [offset, end])
        if hi > lo:
            chosen = batch.take(pa.array(sample_ids[lo:hi]-offset))
            chosen = chosen.append_column('train_row_index', pa.array(sample_ids[lo:hi]))
            selected.append(chosen)
        offset = end
    assert offset == len(meta)
    spectra_sample = pa.Table.from_batches(selected).to_pandas()
    spectra_sample.to_parquet(cache_path, index=False)
    cache_key_path.write_text(json.dumps(sample_key, indent=2), encoding='utf-8')
    print(f'Array sample scan: {time.perf_counter()-started:.1f}s')
assert np.array_equal(spectra_sample['train_row_index'].to_numpy(), sample_ids)
spectra_sample = spectra_sample.drop(columns=meta_columns, errors='ignore').join(
    meta.iloc[sample_ids].reset_index(drop=True))
sample_counts = spectra_sample['ingest_lib'].astype('string').fillna('<missing>').value_counts()
sample_coverage = source_summary[['source', 'spectra']].copy()
sample_coverage['sample n'] = sample_coverage['source'].map(sample_counts).fillna(0).astype(int)
sample_coverage['population %'] = 100*sample_coverage['spectra']/len(meta)
sample_coverage['sample %'] = 100*sample_coverage['sample n']/len(spectra_sample)
assert sample_coverage['sample n'].sum() == len(spectra_sample)
print(f'{len(spectra_sample):,} rows; {len(sample_counts)} sources sampled; cache reused: {cached}')

In [ ]:
coverage_plot = sample_coverage.set_index('source').reindex(source_order)
fig, ax = plt.subplots(figsize=(10.5, 5.5))
y = np.arange(len(coverage_plot))
ax.hlines(y, coverage_plot['population %'], coverage_plot['sample %'], color='#cbd5e1', linewidth=3)
ax.scatter(coverage_plot['population %'], y-.08, color=BLUE, label='Full training metadata', s=45)
ax.scatter(coverage_plot['sample %'], y+.08, color=ORANGE, label='Uniform array sample', s=45)
ax.set(yticks=y, yticklabels=[f"{source}  (sample n={int(coverage_plot.loc[source,'sample n'])})" for source in source_order],
       xlabel='Spectra from source (%)', title='Sample coverage: rare sources need separate follow-up')
ax.invert_yaxis(); ax.legend(loc='lower right'); ax.grid(axis='x', alpha=.15)
show_figure('10_sample_source_coverage', fig)

## 7. Which flags are format errors, and which are attributes to investigate?

Check paired lengths, empty arrays, finite values, sorting, normalization and actual peak counts.
The figure separates **integrity checks** from **spectrum/metadata attributes**. Missing raw intensity, multiple energies or above-precursor peaks do not automatically invalidate a spectrum.
Error bars are approximate Wilson 95% intervals for spectrum-level prevalence under uniform sampling, not molecule-level generalization uncertainty.

In [ ]:
def array_or_none(value):
    if value is None or value is pd.NA or np.isscalar(value):
        return None
    return np.asarray(value, dtype=np.float64)

audit_rows = []
for row in spectra_sample.itertuples(index=False):
    mz = array_or_none(row.ms2_mzs)
    intensity = array_or_none(row.ms2_normalized_intensities)
    ce = array_or_none(row.collision_energy_ev)
    arrays_missing = mz is None or intensity is None
    aligned = not arrays_missing and len(mz) == len(intensity)
    valid = aligned and len(mz) > 0 and np.isfinite(mz).all() and np.isfinite(intensity).all()
    precursor = float(row.precursor_mz) if pd.notna(row.precursor_mz) else np.nan
    retained = {}
    for floor in [0.0, 0.001, 0.01, 0.02]:
        retained[f'retained_floor_{floor:g}'] = int(((mz <= precursor+2) & (intensity >= floor)).sum()) if valid and np.isfinite(precursor) else np.nan
    fingerprint = None
    if valid:
        hasher = hashlib.sha256()
        hasher.update(np.array([precursor], dtype='<f8').tobytes())
        hasher.update(json.dumps([str(row.adduct), str(row.ionization_mode), len(mz)]).encode())
        hasher.update(mz.astype('<f8').tobytes()); hasher.update(intensity.astype('<f8').tobytes())
        fingerprint = hasher.hexdigest()
    audit_rows.append({'train_row_index': row.train_row_index, 'source': str(row.ingest_lib),
        'inchikey14': row.inchikey14, 'adduct': row.adduct, 'precursor_mz': precursor,
        'peak_count': len(mz) if mz is not None else 0,
        'missing_arrays': arrays_missing,
        'empty_arrays': not arrays_missing and (len(mz) == 0 or len(intensity) == 0),
        'length_mismatch': not arrays_missing and not aligned,
        'nonfinite_arrays': not arrays_missing and (not np.isfinite(mz).all() or not np.isfinite(intensity).all()),
        'mz_not_sorted': mz is not None and len(mz)>1 and bool((np.diff(mz)<0).any()),
        'intensity_outside_0_1': intensity is not None and bool(((intensity<0) | (intensity>1+1e-6)).any()),
        'base_peak_not_1': valid and not np.isclose(intensity.max(), 1, atol=1e-6, rtol=0),
        'num_peaks_mismatch': mz is not None and pd.notna(row.num_peaks) and len(mz) != row.num_peaks,
        'has_peak_above_precursor_plus_2': valid and np.isfinite(precursor) and bool((mz>precursor+2).any()),
        'has_nonpositive_mz': mz is not None and bool((mz<=0).any()),
        'base_intensity_missing': pd.isna(row.base_peak_intensity),
        'multi_energy': ce is not None and len(ce)>1,
        'valid_paired_spectrum': valid, 'exact_array_sha256': fingerprint, **retained})
quality_rows = pd.DataFrame(audit_rows)
flags = ['missing_arrays', 'empty_arrays', 'length_mismatch', 'nonfinite_arrays',
         'mz_not_sorted', 'intensity_outside_0_1', 'base_peak_not_1', 'num_peaks_mismatch',
         'has_peak_above_precursor_plus_2', 'has_nonpositive_mz', 'base_intensity_missing', 'multi_energy']
quality_summary = pd.DataFrame({'check': flags, 'sample n': len(quality_rows),
    'flagged n': [int(quality_rows[c].sum()) for c in flags]})
quality_summary['flagged %'] = 100*quality_summary['flagged n']/len(quality_rows)
def wilson_interval(count, total):
    z = 1.96
    p = count/total
    denominator = 1+z*z/total
    center = (p+z*z/(2*total))/denominator
    half = z*np.sqrt(p*(1-p)/total+z*z/(4*total*total))/denominator
    return max(0, center-half)*100, min(1, center+half)*100

intervals = [wilson_interval(n, len(quality_rows)) for n in quality_summary['flagged n']]
quality_summary['Wilson 95% low %'] = [v[0] for v in intervals]
quality_summary['Wilson 95% high %'] = [v[1] for v in intervals]
above_ci_low, above_ci_high = wilson_interval(int(quality_rows['has_peak_above_precursor_plus_2'].sum()), len(quality_rows))
print('Intervals approximate spectrum-level prevalence under uniform row sampling; they are not molecule-level uncertainty.')
quality_by_source = quality_rows.groupby('source')[flags].mean().mul(100)
quality_by_source.insert(0, 'sample n', quality_rows.groupby('source').size())
fingerprint_counts = quality_rows['exact_array_sha256'].dropna().value_counts()
duplicate_rows = quality_rows.loc[quality_rows['exact_array_sha256'].isin(fingerprint_counts.index[fingerprint_counts.gt(1)])]
print(f'Exact stored-array duplicate groups within sample: {int(fingerprint_counts.gt(1).sum())}')
print('Fingerprint includes precursor, adduct, ion mode and both complete arrays; near duplicates are not tested.')
if len(duplicate_rows):
    display(duplicate_rows[['train_row_index', 'source', 'inchikey14', 'exact_array_sha256']].head(8))

valid_quality = quality_rows.loc[quality_rows['valid_paired_spectrum'] & np.isfinite(quality_rows['precursor_mz'])]
retention_rows = []
for floor in [0.0, 0.001, 0.01, 0.02]:
    retained = valid_quality[f'retained_floor_{floor:g}']
    for cap in [None, 128, 256]:
        kept = retained if cap is None else retained.clip(upper=cap)
        retention_rows.append({'relative floor': floor, 'top-N cap': 'none' if cap is None else cap,
            'eligible spectra': len(kept), 'kept peaks p50': kept.median(),
            'kept peaks p95': kept.quantile(.95),
            'kept/original peaks p50 %': 100*(kept/valid_quality['peak_count']).median(),
            'total peaks retained %': 100*kept.sum()/valid_quality['peak_count'].sum(),
            'spectra truncated by cap %': 0 if cap is None else 100*retained.gt(cap).mean(),
            'spectra left with <5 peaks %': 100*kept.lt(5).mean()})
retention_summary = pd.DataFrame(retention_rows)
print('All filter scenarios include m/z <= precursor + 2; no source arrays are changed.')

In [ ]:
labels = {'missing_arrays':'Missing peak arrays','empty_arrays':'Empty arrays','length_mismatch':'Paired lengths differ',
          'nonfinite_arrays':'Nonfinite values','mz_not_sorted':'m/z not sorted','intensity_outside_0_1':'Intensity outside [0,1]',
          'base_peak_not_1':'Maximum intensity differs from 1','num_peaks_mismatch':'Stored peak count differs',
          'has_nonpositive_mz':'Nonpositive m/z','has_peak_above_precursor_plus_2':'Any peak above precursor +2',
          'base_intensity_missing':'Raw base intensity missing','multi_energy':'Multiple collision energies'}
fig, axes = plt.subplots(1,2,figsize=(14,5.2),gridspec_kw={'width_ratios':[1.3,1]})
integrity = [f for f in flags if f not in ['has_peak_above_precursor_plus_2','base_intensity_missing','multi_energy']]
attributes = ['has_peak_above_precursor_plus_2','base_intensity_missing','multi_energy']
for ax, selection, color, title in zip(axes,[integrity,attributes],[BLUE,ORANGE],['Integrity checks in the sample','Attributes: investigate before filtering']):
    stats = quality_summary.set_index('check').loc[selection]
    rate = stats['flagged %'].to_numpy()
    low, high = stats['Wilson 95% low %'].to_numpy(), stats['Wilson 95% high %'].to_numpy()
    y = np.arange(len(stats))
    ax.barh(y,rate,color=color,alpha=.55)
    ax.errorbar(rate,y,xerr=[np.maximum(0,rate-low),np.maximum(0,high-rate)],fmt='o',color=color,capsize=3)
    ax.set(yticks=y,yticklabels=[labels[x] for x in selection],xlabel='Sampled spectra flagged (%)',title=title)
    ax.invert_yaxis(); ax.set_xlim(0,max(1, high.max()*1.35))
    for yi, (_, row) in enumerate(stats.iterrows()):
        ax.text(float(row['Wilson 95% high %'])+ax.get_xlim()[1]*.02,yi,f"{int(row['flagged n'])}/{len(quality_rows)}",va='center',fontsize=8)
show_figure('11_quality_flags_with_uncertainty',fig)

**Read zero counts correctly:** no observed case in the sample does not prove absence in all training rows; the interval retains this uncertainty.
**Read above-precursor flags carefully:** co-isolation, noise, ion charge and metadata are competing explanations.
`+2 Da` is an audit line; review ion forms rather than applying a universal physical verdict. Return to actual examples next.

## 5. Inspect actual examples, not only summary tables

Plot a typical sampled spectrum, the longest sampled spectrum, a spectrum with peaks above precursor,
and the spectrum with the largest absolute precursor error. Mark a category absent if the sample contains none.
Keep original row position, source, connectivity key, energy and precursor error for review.
The red line marks precursor `+2 Da`; plots show paired raw arrays without filtering.

A precursor error in `ppm` is a relative error in parts per million: at m/z 300, 10 ppm corresponds to a difference of 0.003. This training field is computed relative to the known answer and adduct; use it for label/metadata review, not as a ready-made test input.

In [ ]:
eligible = quality_rows.loc[quality_rows['valid_paired_spectrum']]
sample_abs_error = pd.Series(numeric(spectra_sample['precursor_error_ppm']), index=spectra_sample.index).abs()
sample_abs_error = sample_abs_error.loc[eligible.index].replace([np.inf, -np.inf], np.nan).dropna()
choices = [('Typical sampled spectrum', (eligible['peak_count']-eligible['peak_count'].median()).abs().idxmin() if len(eligible) else None),
           ('Longest sampled spectrum', eligible['peak_count'].idxmax() if len(eligible) else None),
           ('Above-precursor example', eligible.index[eligible['has_peak_above_precursor_plus_2']][0] if eligible['has_peak_above_precursor_plus_2'].any() else None),
           ('Largest |precursor error|', sample_abs_error.idxmax() if len(sample_abs_error) else None)]
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
example_records = []
for ax, (reason, idx) in zip(axes.flat, choices):
    if idx is None:
        ax.text(.5, .5, 'No example in this sample', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(reason); continue
    row = spectra_sample.loc[idx]
    mz, intensity = array_or_none(row['ms2_mzs']), array_or_none(row['ms2_normalized_intensities'])
    ax.vlines(mz, 0, intensity, linewidth=.6, color='#2166ac')
    if pd.notna(row['precursor_mz']):
        ax.axvline(float(row['precursor_mz'])+2, color='#d6604d', linestyle='--', linewidth=1)
    ax.set(title=f"{reason} | row {row['train_row_index']} | {row['ingest_lib']}", xlabel='Fragment m/z', ylabel='Relative intensity')
    example_records.append({'reason': reason, **{c: row[c] for c in
        ['train_row_index', 'ingest_lib', 'inchikey14', 'normalized_smiles', 'adduct',
         'precursor_mz', 'num_peaks', 'precursor_error_ppm', 'collision_energy_ev']}})
show_figure('12_real_spectrum_examples', fig)
example_table = pd.DataFrame(example_records)
display(example_table[['reason','train_row_index','adduct','num_peaks','precursor_error_ppm']].round(2))

## 8. What does cleaning keep or discard? See it on one spectrum first

Choose the longest sampled paired spectrum with adduct `[M+H]+` and illustrate a **candidate policy**:
precursor `+2` limit → relative-intensity floor 1% → strongest 128 peaks.
Gray marks every original peak; green marks retained peaks. This shows an input transformation, not an accuracy gain, and leaves source data unchanged.

In [ ]:
illustration_candidates = eligible.loc[eligible['adduct'].eq('[M+H]+') & np.isfinite(eligible['precursor_mz'])]
if illustration_candidates.empty:
    illustration_candidates = eligible.loc[np.isfinite(eligible['precursor_mz'])]
filter_idx = illustration_candidates['peak_count'].idxmax()
filter_row = spectra_sample.loc[filter_idx]
mz = array_or_none(filter_row['ms2_mzs']); intensity = array_or_none(filter_row['ms2_normalized_intensities'])
precursor = float(filter_row['precursor_mz'])
mass_mask = mz <= precursor+2
floor_mask = mass_mask & (intensity >= .01)
remaining_positions = np.flatnonzero(floor_mask)
kept_positions = remaining_positions[np.argsort(intensity[remaining_positions])[::-1][:128]]
filter_example = {'train_row_index':int(filter_row['train_row_index']), 'original_peaks':len(mz),
                  'after_mass_limit':int(mass_mask.sum()), 'after_floor_1pct':int(floor_mask.sum()), 'after_top128':len(kept_positions)}
fig, axes = plt.subplots(1,2,figsize=(13,4.6),sharex=True,sharey=True)
axes[0].vlines(mz,0,intensity,color=BLUE,linewidth=.6)
axes[0].set_title(f"Original: {len(mz):,} peaks | row {filter_row['train_row_index']}")
axes[1].vlines(mz,0,intensity,color='#cbd5e1',linewidth=.6,label='Original peaks')
axes[1].vlines(mz[kept_positions],0,intensity[kept_positions],color=GREEN,linewidth=1.2,label=f'Kept: {len(kept_positions)} peaks')
axes[1].axhline(.01,color=ORANGE,linestyle=':',label='1% intensity floor')
axes[1].set_title('Candidate filter: the gray information is discarded')
axes[1].legend(fontsize=8,loc='upper right')
for ax in axes:
    ax.axvline(precursor+2,color=ORANGE,linestyle='--',linewidth=1)
    ax.set(xlabel='Fragment m/z',ylabel='Relative intensity',ylim=(0,1.1))
show_figure('13_filter_before_and_after',fig)
display(pd.DataFrame([filter_example]))

### Apply the same policy across the sample: medians can hide tail losses

The left panel measures **retained total peaks / original total peaks**; the right measures **spectra left with fewer than 5 peaks / eligible spectra**.
Their denominators differ. Intensity floors and peak caps can remove substantial information; stronger floors can also leave spectra sparse.
Only a comparison under a fixed validation protocol can decide which policy works better.

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(12,4.6))
for cap, color in [('none',BLUE),(128,ORANGE),(256,GREEN)]:
    rows = retention_summary.loc[retention_summary['top-N cap'].eq(cap)].sort_values('relative floor')
    axes[0].plot(np.arange(len(rows)),rows['total peaks retained %'],'o-',color=color,label=f'top-N: {cap}')
uncapped = retention_summary.loc[retention_summary['top-N cap'].eq('none')].sort_values('relative floor')
axes[1].plot(np.arange(len(uncapped)),uncapped['spectra left with <5 peaks %'],'o-',color=ORANGE)
for x,y in zip(np.arange(len(uncapped)),uncapped['spectra left with <5 peaks %']):
    axes[1].annotate(f'{y:.1f}%',(x,y),xytext=(0,8),textcoords='offset points',ha='center',fontsize=9)
axes[0].set(xlabel='Relative-intensity floor (settings, %)',ylabel='Original total peaks retained (%)',title='Peak information removed by each policy',ylim=(0,105))
axes[0].legend(fontsize=9)
axes[1].set(xlabel='Relative-intensity floor (settings, %)',ylabel='Eligible spectra with <5 peaks (%)',title='How often does the input become sparse?',ylim=(0,max(uncapped['spectra left with <5 peaks %'])*1.25))
for ax in axes:
    ax.set_xticks(range(4),['0','0.1','1','2']); ax.grid(alpha=.15)
show_figure('14_filter_sensitivity',fig)

## 9. Connect the figures to action: define evaluation before choosing inputs

We moved from one spectrum to molecular groups, source conditions and filtering effects.
Observations are not performance results: structure overlap changes evaluation, sources motivate slices, quality flags require review, and retention plots measure information preserved.
Save figures, tables, sample row positions and package versions, then read the short decision summary.

In [ ]:
from datetime import datetime, timezone
import matplotlib

tables = {'group_summary': group_summary, 'label_examples': label_examples, 'formula_examples': formula_examples,
          'source_summary': source_summary, 'source_instrument_mix': instrument_mix.reset_index(),
          'source_energy_units': energy_units.reset_index(),
          'conditional_peak_counts': conditional_peaks.reset_index(), 'sample_coverage': sample_coverage,
          'quality_summary': quality_summary, 'quality_by_source': quality_by_source.reset_index(),
          'sample_quality_rows': quality_rows, 'filter_retention': retention_summary,
          'example_spectra': example_table, 'sample_duplicate_rows': duplicate_rows}
for name, frame in tables.items():
    frame.to_csv(OUTPUT_DIR / f'{name}.csv', index=False)

findings = {
    'train_rows': len(meta), 'train_smiles_labels': len(train_groups),
    'train_connectivity_groups': int(train_connectivity_groups),
    'multi_source_groups_pct': float(multi_source_pct),
    'label_variant_groups': int(group_audit['smiles_labels'].gt(1).sum()),
    'formula_variant_groups': int(group_audit['formulas'].gt(1).sum()),
    'iid_row_split_same_structure_partner_pct': float(row_split_partner_pct),
    'np_example_groups': len(np_keys), 'np_example_groups_in_other_sources': np_overlap_n,
    'sample_rows': len(quality_rows), 'sample_sources': len(sample_counts),
    'sample_above_precursor_pct': float(quality_rows['has_peak_above_precursor_plus_2'].mean()*100),
    'sample_above_precursor_wilson_95_pct': [float(above_ci_low), float(above_ci_high)],
    'sample_exact_duplicate_groups': int(fingerprint_counts.gt(1).sum()),
    'visible_test_role': 'placeholder; interface checks only; not hidden-test evidence',
}
manifest = {'created_utc': datetime.now(timezone.utc).isoformat(),
    'inputs': input_identity, 'seed': SEED, 'sample_n': len(quality_rows),
    'sample_method': 'uniform rows without replacement over the complete training file',
    'sample_row_indices_sha256': hashlib.sha256(sample_ids.astype('<i8').tobytes()).hexdigest(),
    'identity_note': 'Parquet footer SHA256 is a metadata digest, not a whole-file content hash.',
    'versions': {'python': platform.python_version(), 'pandas': pd.__version__,
                 'numpy': np.__version__, 'pyarrow': pa.__version__, 'matplotlib': matplotlib.__version__},
    'sources': ['https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/data',
                'https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/overview'],
    'limitations': ['No full-array quality audit or full-dataset near-duplicate search.',
        'Supplied connectivity keys have not been recomputed using official normalization.',
        'Conditional source comparisons are observational, not causal.',
        'No training, scoring, cleaning, or transductive test adaptation performed.'],
    'findings': findings}
manifest['visuals'] = figure_records
manifest['visual_revision'] = 'v3-beginner-story'
manifest['rdkit_depictions_available'] = HAS_RDKIT
if HAS_RDKIT:
    import rdkit
    manifest['versions']['rdkit'] = rdkit.__version__
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('Evidence directory:', OUTPUT_DIR.resolve())
missing_matrix.reset_index().to_csv(OUTPUT_DIR / 'source_missingness.csv', index=False)
pd.DataFrame([filter_example]).to_csv(OUTPUT_DIR / 'filter_example.csv', index=False)
(OUTPUT_DIR / 'figure_manifest.json').write_text(json.dumps(figure_records, indent=2), encoding='utf-8')

In [ ]:
from IPython.display import Markdown

report = f'''# CASMI 2026 EDA: evidence and next decisions

Training file: **{len(meta):,} spectra**, **{len(train_groups):,} SMILES labels**, **{train_connectivity_groups:,} supplied connectivity keys**.
Array audit: **{len(quality_rows):,} uniformly sampled whole-file rows**, seed **{SEED}**.

| Observation and evidence | Possible explanation | Validation/model implication | Cheapest next check |
|:--|:--|:--|:--|
| {multi_source_pct:.2f}% of connectivity groups span sources; an IID 80/20 row allocation gives an expected {row_split_partner_pct:.2f}% same-structure training-partner rate for validation rows (group_summary.csv) | Repeated molecular measurements, not necessarily identical spectra | Specify known-structure retrieval versus unseen-structure generalization; row holdouts can share structures | Build structure-group indices and inspect source overlaps; no actual split exists yet |
| {int(group_audit['smiles_labels'].gt(1).sum()):,} groups have multiple SMILES; {int(group_audit['formulas'].gt(1).sum()):,} have multiple formulas (label_examples.csv; formula_examples.csv) | Stereochemistry, tautomers, protonation or metadata variation; displayed formulas differ in charge state, not established mislabeling | Deduplicate candidates with the official normalized key; avoid automatic relabeling | Normalize the displayed examples with the official pinned implementation and check scoring equivalence |
| {np_overlap_n} of {len(np_keys)} enveda-np-examples keys also occur in other libraries (source_summary.csv; section code) | Other sources contain reference spectra for these structures | A source holdout can retain the same structure; instrument matching does not ensure chemistry matching | Document allowed/excluded references and compare natural-product and other-source slices |
| {quality_rows['has_peak_above_precursor_plus_2'].mean()*100:.2f}% of sampled spectra contain peaks above precursor +2 Da; approximate spectrum-level 95% interval {above_ci_low:.2f}–{above_ci_high:.2f}% (quality_summary.csv; example_spectra.csv) | Noise, co-isolation, ion charge or metadata issues | Review filtering by source and ion type | Inspect displayed examples, then compare filters on a fixed validation set |
| {int(fingerprint_counts.gt(1).sum())} complete-array exact duplicate groups within the sample (sample_duplicate_rows.csv) | Official exact deduplication and/or limited sample coverage | Does not establish absence of near duplicates across all training rows | Hash full arrays and screen near duplicates at a selected holdout boundary |

**Filter sensitivity only:** `filter_retention.csv` compares intensity floors and 128/256 peak caps; it measures retained information, not accuracy.
**Visible test boundary:** interface and molecule-group checks only. It is a placeholder, not evidence about hidden-test distributions or leaderboard scores.
**Unknowns:** official-normalization key equivalence, independent annotation confidence, acquisition chronology, an implemented validation protocol and model scores.

Evidence directory: `{OUTPUT_DIR.resolve()}`. Input identity and package versions are in `manifest.json`.
Prioritize a molecule/structure evaluation protocol before choosing cleaning rules or peak budgets.
'''
(OUTPUT_DIR / 'report_en.md').write_text(report, encoding='utf-8')

display(Markdown(f'''
### Four conclusions to take away

- **Think in molecules first.** Training has {len(meta):,} spectra and {train_connectivity_groups:,} supplied connectivity keys; one row is not an independent molecule.
- **A source holdout still needs structure checks.** {multi_source_pct:.2f}% of structure groups span sources; {np_overlap_n}/{len(np_keys)} natural-product example keys also appear elsewhere.
- **Inspect flags before cleaning.** The sampled above-precursor rate is {quality_rows['has_peak_above_precursor_plus_2'].mean()*100:.2f}% (approximate 95% interval {above_ci_low:.2f}–{above_ci_high:.2f}%); it is not an automatic deletion list.
- **Filtering is a tradeoff.** Figures measure retention and sparsity. The cheapest next step is structure-aware evaluation before comparing filter policies.

Saved **{len(figure_records)} figure groups**. Complete evidence, technical reports and CSVs are in `{OUTPUT_DIR}`.
Visible test is an interface example; this EDA contains no model accuracy result.
'''))

<details><summary>Optional appendix: field-by-field reference</summary>

### Every column: can it enter a model?

**Rule: an input feature must be obtainable the same way in train and test.** “Shared” does not mean mandatory; arrays, categories, and missing values still need preprocessing. The meanings follow the [official field descriptions](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/data).

**Ten shared measurement columns**

| Column | Meaning | Model use |
|:--|:--|:--|
| `ms2_mzs` | Array of fragment m/z values | **Primary input**; pair by index with intensities and handle variable length. |
| `ms2_normalized_intensities` | Aligned relative intensities; maximum is 1 | **Primary input**; never sort separately from m/z. |
| `precursor_mz` | Measured precursor-ion m/z | Usable; interpret with adduct, not as neutral mass. |
| `adduct` | Standardized ion form, e.g. `[M+H]+` | Usable category. |
| `ionization_mode` | Positive or negative ionization | Usable category. |
| `instrument_type` | Instrument label | Optional; free-text train labels and a narrower test instrument mix require care. |
| `collision_energy_ev` | **List** of collision energies in eV; multiple values may reflect merged acquisitions | Optional; handle missing and multiple values. |
| `collision_energy_orig` | Energy text exactly as recorded by its source | Parse with units before use; do not treat mixed-unit strings as one numeric scale. |
| `collision_energy_orig_units` | Units of the original energy, e.g. eV, NCE, V | Helps parse original energy; often unnecessary if using `collision_energy_ev`. |
| `base_peak_intensity` | Raw strongest-peak intensity before normalization | Optional; missing in some training libraries and not a concentration measure. |

**Eight train-only columns: not ready-made test inputs**

| Column | Meaning | Model use |
|:--|:--|:--|
| `normalized_smiles` | Known standardized structure | **Main target `y`**, never part of input `X`. |
| `inchikey` | Full identifier of the known structure | Deduplication or label checks; not a test input. |
| `inchikey14` | First InChIKey block, grouping 2D connectivity | Grouped validation or auxiliary target; not a test input. |
| `molecular_formula` | Atom counts of the known neutral structure | Auxiliary target or candidate property; the stored train value is not a test input. |
| `ingest_lib` | Source library of a training spectrum | Source-bias audit or evaluation slice; no matching test field. |
| `adduct_orig` | Source's original adduct notation | Audit standardization; use shared `adduct` at inference. |
| `precursor_error_ppm` | Difference between measured precursor and value implied by the **known structure plus adduct** | Label-quality check; raw value leaks the answer. Recompute per candidate if building a candidate ranker. |
| `num_peaks` | Fragment peak count | **Usable**, if computed on both sides as `len(ms2_mzs)`. |

**Two test-only columns: manage predictions, not chemical features**

| Column | Meaning | Model use |
|:--|:--|:--|
| `molecule_id` | Group ID of an unknown molecule | Combine its spectra and write one submission row; not a predictive feature. |
| `spectrum_id` | Unique ID of one test spectrum | Track records; not a predictive feature. |

</details>